# 🗺️ GPHY 284 — Nominal, Ordinal, Interval, Ratio: Which Scale Is It?
Run each cell top to bottom (Shift+Enter). Your score is kept as you go. All data here is **synthetic/illustrative**. These aren't real statistics.

#### Run this cell to connect to your GIS and get started:

In [ ]:
from arcgis.gis import GIS
gis = GIS("home")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    import ipywidgets as w; from IPython.display import display; WIDGETS=True
except ImportError:
    WIDGETS=False
SCORE={'right':0,'tried':0}
def score(): print(f"🏆 Score: {SCORE['right']}/{SCORE['tried']}")
print('Widgets available:', WIDGETS)

| Scale | Bozeman/Montana example | Math allowed |
|---|---|---|
| **Nominal**: names/categories | Land cover: forest, grassland, urban (Gallatin Valley) | `=` / `≠` only: counts and the mode |
| **Ordinal**: ranked categories | Fire danger: Low → Moderate → High → Very High → Extreme | `<` `>`: median, ranks (but the gaps aren't equal!) |
| **Interval**: equal steps, *arbitrary zero* | Temperature in °F/°C at the Bridger Bowl base | `+` `−`: differences and means, **but no ratios** |
| **Ratio**: equal steps, *true zero* | Snow depth (cm), population, distance to Bozeman | everything, including `×` `÷` ("twice as much" works) |

## Exercise 1: Classify the attribute 🃏 (8 cards)
Pick a scale for each card. Without widgets, call `answer(card_number, 'ratio')` instead.

In [ ]:
CARDS=[
 ('ZIP code (59715)','nominal','A label. Averaging ZIP codes gives you nonsense.'),
 ('Avalanche danger (Low → Extreme)','ordinal','Ranked, but Considerable−Moderate ≠ High−Considerable.'),
 ('Snow depth (cm)','ratio','0 cm means no snow, so 100 cm really is twice 50 cm.'),
 ('Air temperature (°F)','interval','Equal steps, but 0°F is arbitrary, so no ratios.'),
 ('Road type (interstate, highway, gravel)','nominal','Categories. (If you rank them by size it becomes ordinal, so context matters!)'),
 ('Hurricane category (1–5)','ordinal','Numbered ranks, and the wind-speed gaps between categories aren\'t equal.'),
 ('Home value ($)','ratio','$0 is a true zero, so a $600k house costs twice a $300k one.'),
 ('County FIPS code (30031 = Gallatin)','nominal','Numbers used as IDs. Never do math on them.'),
]
SCALES=['nominal','ordinal','interval','ratio']
def answer(i,guess):
    name,ans,why=CARDS[i-1]; SCORE['tried']+=1
    if guess.strip().lower()==ans: SCORE['right']+=1; print(f'✅ Card {i}: {name}. Correct, it\'s {ans}! {why}')
    else: print(f'❌ Card {i}: {name}. It\'s {ans.upper()}. {why}')
    score()
if WIDGETS:
    for i,(name,_,_) in enumerate(CARDS,1):
        out=w.Output(); b=w.ToggleButtons(options=SCALES,value=None,description=f'{i}. {name}',style={'description_width':'initial'})
        def cb(ch,i=i,out=out,b=b):
            if ch['new'] and not b.disabled:
                with out: answer(i,ch['new'])
                b.disabled=True
        b.observe(cb,names='value'); display(w.VBox([b,out]))
else:
    for i,(n,_,_) in enumerate(CARDS,1): print(f'{i}. {n}')
    print("Fallback: e.g. answer(1,'nominal')")

## Exercise 2: Squash ratio data into ordinal classes 📉
Synthetic seasonal snowfall (inches) for 12 made-up Montana stations. Compare **equal-interval** and **quantile** breaks.

In [ ]:
rng=np.random.default_rng(284)
snow=pd.DataFrame({'station':[f'Stn-{c}' for c in 'ABCDEFGHIJKL'],
                   'snow_in':np.round(np.r_[rng.gamma(2,40,10),[420,510]],0)})  # 2 skewed 'ski-area' stations
labels=['Low','Medium','High']
snow['equal_interval']=pd.cut(snow.snow_in,3,labels=labels)
snow['quantile']=pd.qcut(snow.snow_in,3,labels=labels)
print(snow.sort_values('snow_in').to_string(index=False))
print('\nCounts per class:'); print(pd.DataFrame({'equal':snow.equal_interval.value_counts(),'quantile':snow['quantile'].value_counts()}).loc[labels])

**🤔 Quick question:** Two stations that differ by ~100 inches can end up in the same class. That's **information lost**: once data is ordinal you can rank it, but you can't do arithmetic on it anymore. Which method lumps almost everything into 'Low'? Why?

Answer here:

## Exercise 3: Is 20°F twice as warm as 10°F? 🌡️

In [ ]:
def f_to_k(f): return (f-32)*5/9+273.15
f1,f2=10,20
print(f'°F ratio: {f2}/{f1} = {f2/f1:.2f}  ← meaningless, because 0°F is arbitrary')
print(f'°C ratio: {(f2-32)*5/9:.1f}/{(f1-32)*5/9:.1f} = {((f2-32)*5/9)/((f1-32)*5/9):.2f}  ← different answer, same weather!')
print(f'Kelvin:   {f_to_k(f2):.1f}/{f_to_k(f1):.1f} = {f_to_k(f2)/f_to_k(f1):.3f}  ← true zero, so the ratio is valid (about 2% warmer)')

In [ ]:
# Your turn: is a 40°F afternoon "twice as warm" as a 20°F morning? Edit and run.
my_ratio = f_to_k(40)/f_to_k(20)
print(f'Kelvin ratio: {my_ratio:.3f}', '→ nope, not twice!' if my_ratio<1.5 else '')

## Exercise 4: Spot the bad map 🚨
Both panels show the same **nominal** land-cover grid (synthetic). Which one is lying to you?

In [ ]:
rng=np.random.default_rng(1)
grid=rng.integers(0,4,(12,12)); names=['Forest','Grassland','Urban','Water']
fig,ax=plt.subplots(1,2,figsize=(9,4))
for a,cm,t in [(ax[0],'viridis','Map A'),(ax[1],'tab10','Map B')]:
    im=a.imshow(grid,cmap=cm,vmin=0,vmax=(3 if cm=='viridis' else 9)); a.set_title(t); a.axis('off')
fig.colorbar(ax[0].images[0],ax=ax[0],ticks=range(4)).ax.set_yticklabels(names)
from matplotlib.patches import Patch
ax[1].legend(handles=[Patch(color=plt.cm.tab10(i),label=n) for i,n in enumerate(names)],loc='upper left',bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()

In [ ]:
def bad_map(choice):
    SCORE['tried']+=1
    if choice.upper()=='A': SCORE['right']+=1; print('✅ Right! A sequential ramp makes Water look like "more" than Forest. Nominal data needs distinct categorical colors.')
    else: print('❌ Map B is fine (categorical colors). Map A implies an order that doesn\'t exist.')
    score()
#bad_map('')   # ← change to your answer and re-run